In [ ]:
# =============================
# Logistic Regression (class weights) for your FE dataset
# =============================
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, roc_auc_score,
    classification_report, confusion_matrix
)

In [ ]:

df_train = pd.read_csv('../data/FEwithMerchants/FE_train_with_merchants.csv')
df_val = pd.read_csv('../data/FEwithMerchants/FE_validation_with_merchants.csv')
df_test = pd.read_csv('../data/FEwithMerchants/FE_test_with_merchants.csv')

# df_train = pd.read_csv('../data/FEwithoutMerchants/FE_train_without_merchants.csv')
# df_val = pd.read_csv('../data/FEwithoutMerchants/FE_validation_without_merchants.csv')
# df_test = pd.read_csv('../data/FEwithoutMerchants/FE_test_without_merchants.csv')

print("Converting categorical columns ...")
categorical_cols = [
    'type', 'hourOfDay', 'dayOfWeek', 'transaction_sequence'
]

for col in categorical_cols:
    # Combine categories from both train, val and test, to avoid unseen categories during inference 
    combined_cats = pd.Series(
        pd.concat([df_train[col], df_val[col], df_test[col]], axis=0).dropna().unique()
    )
    categories = sorted(combined_cats.unique())

    df_train[col] = pd.Categorical(df_train[col], categories=categories)
    df_val[col]   = pd.Categorical(df_val[col], categories=categories)
    df_test[col]  = pd.Categorical(df_test[col], categories=categories)

selected_features = [
    # transaction-level features
    'step', 'type', 'hourOfDay', 'day', 'amountLog', 
    'dayOfWeek', 
    # 'amount',         
                
    # account-level features
    'meanSent',
    'transaction_sequence',
    'totalSent', 'stdSent', 'numSent', 
    'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'maxAmountReceived', 
    'stdAmountReceived', 'avgAmountToDest', 'std_to_mean_ratio', 
    'pctForwarded24h',

    # transaction pattern features
    'transactionRecency', 'is_early_transaction',  'sequence_frequency', 'sequence_count',
    'is_transfer_cashout', 'is_cashin_transfer', 'is_cashout_transfer', 'is_cashin_transfer_cashout',
    'is_transfer_transfer', 'is_first_transfer', 'is_cashin_cashout', 

    # aggregated risk signals
    'typeHighValueFlag', 

    # centrality features
    "pagerank_diff", "receiver_indeg_amt",
    "sender_pr","receiver_pr", 
    "sender_outdeg_amt","sender_indeg_amt", "receiver_outdeg_amt",
    "sender_outdeg_cnt","sender_indeg_cnt", "receiver_outdeg_cnt","receiver_indeg_cnt",
    "outdeg_amt_diff","indeg_amt_diff", "outdeg_cnt_diff","indeg_cnt_diff",
]
print("Processing training set ...")
Xtrain, ytrain = df_train[selected_features].copy(), df_train["isFraud"].astype(int)

print("Processing validation set ...")
Xval, yval = df_val[selected_features].copy(), df_val["isFraud"].astype(int)

print("Processing test set ...")
Xtest, ytest = df_test[selected_features].copy(), df_test["isFraud"].astype(int)

print("Done!")

In [ ]:


# ----- Use your existing DataFrames and feature list -----
# df_train, df_val, df_test already loaded above
# selected_features already defined above
# categorical_cols already defined above in your notebook:
# categorical_cols = ['type', 'hourOfDay', 'dayOfWeek', 'transaction_sequence']

# Split features/labels
Xtrain = df_train[selected_features].copy()
ytrain = df_train["isFraud"].astype(int).copy()

Xval = df_val[selected_features].copy()
yval = df_val["isFraud"].astype(int).copy()

Xtest = df_test[selected_features].copy()
ytest = df_test["isFraud"].astype(int).copy()

# Identify numeric vs categorical
num_cols = [c for c in selected_features if c not in categorical_cols]
cat_cols = categorical_cols

# Preprocess: scale nums + one-hot cats (sparse)
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=True, with_std=True), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

def fit_eval_lr(C, class_weight, Xtr, ytr, Xva, yva):
    """Fit LR with given hyperparams; tune threshold on PR curve to maximize F1 (fraud)."""
    # Use solver that works with sparse + class_weight
    lr = LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver="saga",          # supports l1/l2/elasticnet & sparse matrices
        penalty="l2",
        C=C,
        class_weight=class_weight
    )
    pipe = Pipeline([("prep", preprocess), ("clf", lr)])
    pipe.fit(Xtr, ytr)

    # Validation probs and PR metrics
    p = pipe.predict_proba(Xva)[:, 1]
    pr_auc = average_precision_score(yva, p)
    roc_auc = roc_auc_score(yva, p)

    # Threshold tuning: maximize F1 on fraud
    prec, rec, thr = precision_recall_curve(yva, p)
    f1 = 2 * (prec[:-1] * rec[:-1]) / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    best_idx = int(np.nanargmax(f1))
    best_thr = float(thr[best_idx])
    best_f1  = float(f1[best_idx])

    return {
        "pipe": pipe,
        "C": C,
        "class_weight": class_weight,
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_best_thr": best_thr,
        "val_best_f1": best_f1
    }

# ----- Small hyperparameter search (you can expand) -----
C_grid = [0.1, 0.5, 1.0, 2.0, 5.0]
cw_grid = ["balanced", {0:1, 1:5}, {0:1, 1:8}, {0:1, 1:10}, {0:1, 1:15}, {0:1, 1:20}]

results = []
for C in C_grid:
    for cw in cw_grid:
        res = fit_eval_lr(C, cw, Xtrain, ytrain, Xval, yval)
        results.append(res)
        print(f"C={C:<4} cw={cw}  |  PR-AUC={res['val_pr_auc']:.4f}  "
              f"Best F1={res['val_best_f1']:.4f} @ thr={res['val_best_thr']:.3f}")

# Pick the configuration with highest **validation F1** (fraud) after threshold tuning
best = max(results, key=lambda r: r["val_best_f1"])
print("\n=== Best on validation ===")
print(f"C={best['C']}  class_weight={best['class_weight']}")
print(f"Val PR-AUC={best['val_pr_auc']:.4f}, Val ROC-AUC={best['val_roc_auc']:.4f}")
print(f"Chosen threshold (val)={best['val_best_thr']:.4f}, Best F1={best['val_best_f1']:.4f}")

# ----- Refit on train+val with best hyperparams -----
Xtrain_full = pd.concat([Xtrain, Xval], axis=0)
ytrain_full = pd.concat([ytrain, yval], axis=0)

final_lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver="saga",
    penalty="l2",
    C=best["C"],
    class_weight=best["class_weight"]
)
final_pipe = Pipeline([("prep", preprocess), ("clf", final_lr)])
final_pipe.fit(Xtrain_full, ytrain_full)

# ----- Evaluate on test -----
proba_test = final_pipe.predict_proba(Xtest)[:, 1]
pr_auc_test = average_precision_score(ytest, proba_test)
roc_auc_test = roc_auc_score(ytest, proba_test)

# 1) Use tuned threshold from validation
thr_tuned = best["val_best_thr"]
ypred_tuned = (proba_test >= thr_tuned).astype(int)

print("\n=== TEST @ tuned threshold ===")
print(f"Threshold: {thr_tuned:.4f}")
print("PR-AUC :", pr_auc_test)
print("ROC-AUC:", roc_auc_test)
print(classification_report(ytest, ypred_tuned, digits=4))
print("Confusion matrix:\n", confusion_matrix(ytest, ypred_tuned))

# 2) Also show default 0.50 for reference
ypred_05 = (proba_test >= 0.50).astype(int)
print("\n=== TEST @ threshold = 0.50 (reference) ===")
print(classification_report(ytest, ypred_05, digits=4))
